In [ ]:
# 시험 제공용
import pandas as pd
# a = pd.read_csv("../input/traval-insurance-exam/train.csv")
# b = pd.read_csv('../input/traval-insurance-exam/test.csv')

# pd.DataFrame(변수).to_csv('000000000.csv')

# 보험가입 확률을 묻는 문제


# 출력형태 
# index,y_pred
# 0,0.15246735144689827
# 1,0.9615112924356346
# 2,0.15131398160778645
# 3,0.23663288296712173
# 4,0.9385824940846587
# 5,0.15738583559845978


## 풀이
- 아래 풀이는 예시 일 뿐임

In [ ]:
train = pd.read_csv("../input/traval-insurance-exam/train.csv")
test = pd.read_csv('../input/traval-insurance-exam/test.csv')

## EDA

In [ ]:
# 크기 확인
train.shape, test.shape

In [ ]:
# 타입 확인
train.info()

In [ ]:
# 결측치 확인
train.isnull().sum()

In [ ]:
# 결측치 확인
test.isnull().sum()

In [ ]:
train.describe()

In [ ]:
train['TravelInsurance'].value_counts()

## 데이터 전처리

In [ ]:
# 수치형 데이터와 범주형 데이터 분리 
n_train = train.select_dtypes(exclude='object').copy()
c_train = train.select_dtypes(include='object').copy()
n_test = test.select_dtypes(exclude='object').copy()
c_test = test.select_dtypes(include='object').copy()

In [ ]:
# 수치형 변수
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
cols = ['Age', 'AnnualIncome', 'FamilyMembers', 'ChronicDiseases']

display(n_train.head())
n_train[cols] = scaler.fit_transform(n_train[cols])
n_test[cols] = scaler.transform(n_test[cols])
n_train.head()

In [ ]:
# 범주형 변수
display(c_train.head())
c_train = pd.get_dummies(c_train)
c_test = pd.get_dummies(c_test)
c_train.head()

In [ ]:
# 분리한 데이터 다시 합침
train = pd.concat([n_train, c_train], axis=1)
test = pd.concat([n_test, c_test], axis=1)
print(train.shape, test.shape)
train.head()

## 검증 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train.drop('TravelInsurance', axis=1), 
                                            train['TravelInsurance'],
                                            test_size=0.1,
                                            random_state=1204)
X_tr.shape, X_val.shape, y_tr.shape, y_val.shape

## 모델

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=400, max_depth=9, random_state=1204)
rf.fit(X_tr, y_tr)
pred = rf.predict_proba(X_val)[:,1]

## 검증 평가

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_val, pred)

## test 데이터 예측

In [ ]:
pred = rf.predict_proba(test)[:,1]

In [ ]:
# csv 파일 생성 (예시와 다른 형태)
pd.DataFrame({'index':test.index,'y_pred':pred}).to_csv('0000.csv', index=False)

In [ ]:
# 또는
pd.DataFrame({'y_pred':pred}).reset_index().to_csv('1111.csv', index=False)

In [ ]:
pd.read_csv("./0000.csv")

In [ ]:
pd.read_csv("./1111.csv")

## 평가(체점)

In [ ]:
y_test = pd.read_json("../input/traval-insurance-exam/y_test.json")
roc_auc_score(y_test, pred)